# UCF-Crime GPT Video Description Generation

This notebook generates one global GPT description per UCF-Crime video from UCA timestamp-level annotations.


## 1. Config

Paste the OpenAI API key directly below when you are ready to call the model. Keep `DRY_RUN = True` to test data loading without spending API calls.


In [ ]:
from pathlib import Path
import csv
import json
import random
import re
import time
from collections import Counter, defaultdict

try:
    import requests
except ImportError as exc:
    raise ImportError("Please install requests first, for example: pip install requests") from exc

OPENAI_API_KEY = ""
OPENAI_MODEL = "gpt-5.6-luna"

# Choose which splits to generate. Use ["train"] for fine-tuning only,
# or ["train", "val", "test"] to generate all UCA annotations.
SPLITS_TO_PROCESS = ["train", "val", "test"]

DRY_RUN = False
REQUEST_TIMEOUT_SECONDS = 180
SLEEP_BETWEEN_REQUESTS_SECONDS = 0.25
OPENAI_MAX_OUTPUT_TOKENS = 500
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 1.5
RERUN_WARNINGS_ONLY = True


current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / "UCF Annotation").exists() else current_dir.parent
CODE_DIR = PROJECT_ROOT / "code"
UCA_JSON_BY_SPLIT = {
    "train": PROJECT_ROOT / "UCF Annotation" / "json" / "UCFCrime_Train.json",
    "val": PROJECT_ROOT / "UCF Annotation" / "json" / "UCFCrime_Val.json",
    "test": PROJECT_ROOT / "UCF Annotation" / "json" / "UCFCrime_Test.json",
}
OUTPUT_JSON = CODE_DIR / "ucf_gpt_video_descriptions.json"
OUTPUT_CSV = CODE_DIR / "ucf_gpt_video_descriptions.csv"

print("Project root:", PROJECT_ROOT)
print("Splits:", SPLITS_TO_PROCESS)
print("Dry run:", DRY_RUN)
print("Rerun warnings only:", RERUN_WARNINGS_ONLY)


Project root: D:\Finetune VadCLIP
Splits: ['train', 'val', 'test']
Dry run: False
Rerun warnings only: True


## 2. Load And Normalize UCA Annotations


In [2]:
UCF_CRIME_CLASSES = [
    "Abuse",
    "Arrest",
    "Arson",
    "Assault",
    "Burglary",
    "Explosion",
    "Fighting",
    "Normal",
    "RoadAccidents",
    "Robbery",
    "Shooting",
    "Shoplifting",
    "Stealing",
    "Vandalism",
]


def normalize_text_artifacts(text: str) -> str:
    replacements = {
        "‘": "'",
        "’": "'",
        "“": '"',
        "”": '"',
        "–": "-",
        "—": "-",
        "�": "'",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def infer_class_name(video_id: str) -> str:
    if video_id.startswith("Normal"):
        return "Normal"
    match = re.match(r"^([A-Za-z]+)", video_id)
    if not match:
        return "Unknown"
    return match.group(1)


def clean_sentence(sentence: str) -> str:
    sentence = normalize_text_artifacts(sentence)
    sentence = re.sub(r"\s+", " ", sentence.strip())
    sentence = sentence.replace(" .", ".")
    return sentence


def normalize_for_dedup(sentence: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", sentence.lower()).strip()


def sort_events(timestamps, sentences):
    events = []
    for ts, sentence in zip(timestamps, sentences):
        start = float(ts[0]) if len(ts) > 0 else 0.0
        end = float(ts[1]) if len(ts) > 1 else start
        events.append({"timestamp": [start, end], "sentence": clean_sentence(sentence)})
    return sorted(events, key=lambda item: (item["timestamp"][0], item["timestamp"][1]))


def build_raw_joined_description(events):
    kept = []
    previous_norm = None
    for event in events:
        sentence = event["sentence"]
        sentence_norm = normalize_for_dedup(sentence)
        if sentence and sentence_norm != previous_norm:
            kept.append(sentence)
        previous_norm = sentence_norm
    return " ".join(kept)


def load_records_for_split(split_name):
    json_path = UCA_JSON_BY_SPLIT[split_name]
    with json_path.open("r", encoding="utf-8") as f:
        split_data = json.load(f)

    split_records = []
    for video_id, payload in split_data.items():
        events = sort_events(payload.get("timestamps", []), payload.get("sentences", []))
        split_records.append(
            {
                "split": split_name,
                "video_id": video_id,
                "class_name": infer_class_name(video_id),
                "duration": float(payload.get("duration", 0.0)),
                "timestamps": [event["timestamp"] for event in events],
                "source_sentences": [event["sentence"] for event in events],
                "raw_joined_description": build_raw_joined_description(events),
            }
        )
    return split_records


records = []
for split_name in SPLITS_TO_PROCESS:
    records.extend(load_records_for_split(split_name))

print("Loaded videos:", len(records))
print("Split counts:", dict(Counter(record["split"] for record in records)))
print("Class counts:")
for class_name, count in sorted(Counter(record["class_name"] for record in records).items()):
    print(f"  {class_name}: {count}")


Loaded videos: 1854
Split counts: {'train': 1165, 'val': 379, 'test': 310}
Class counts:
  Abuse: 50
  Arrest: 50
  Arson: 50
  Assault: 48
  Burglary: 100
  Explosion: 50
  Fighting: 50
  Normal: 910
  RoadAccidents: 148
  Robbery: 149
  Shooting: 50
  Shoplifting: 50
  Stealing: 100
  Vandalism: 49


## 3. Select Videos To Process


In [3]:
split_order = {split_name: idx for idx, split_name in enumerate(SPLITS_TO_PROCESS)}
records_to_process = sorted(records, key=lambda item: (split_order[item["split"]], item["class_name"], item["video_id"]))

print("Videos to process:", len(records_to_process))
print("Split counts:", dict(Counter(record["split"] for record in records_to_process)))

for record in records_to_process[:8]:
    print(record["split"], record["class_name"], record["video_id"], "sentences=", len(record["source_sentences"]))


Videos to process: 1854
Split counts: {'train': 1165, 'val': 379, 'test': 310}
train Abuse Abuse001_x264 sentences= 9
train Abuse Abuse002_x264 sentences= 22
train Abuse Abuse003_x264 sentences= 28
train Abuse Abuse004_x264 sentences= 64
train Abuse Abuse005_x264 sentences= 12
train Abuse Abuse006_x264 sentences= 38
train Abuse Abuse007_x264 sentences= 14
train Abuse Abuse008_x264 sentences= 29


## 4. Prompt And Validation Rules


In [4]:
FORBIDDEN_LABELS = [
    "abuse",
    "arrest",
    "arson",
    "assault",
    "burglary",
    "explosion",
    "fighting",
    "road accident",
    "roadaccident",
    "robbery",
    "shooting",
    "shoplifting",
    "stealing",
    "vandalism",
    "anomaly",
    "anomalous",
    "crime",
]

SYSTEM_PROMPT = """You summarize timestamp-level video descriptions into one global video description for representation-learning supervision.
You must be faithful to the provided descriptions only.
Do not infer causes, identities, intentions, labels, or events that are not explicitly described.
Do not mention anomaly category names or dataset labels.
Prefer neutral wording such as person, people, individual, vehicle, object, and area.
Use plain ASCII punctuation only.
Return only valid JSON."""


def make_user_prompt(record):
    numbered_events = []
    for idx, (timestamp, sentence) in enumerate(zip(record["timestamps"], record["source_sentences"]), start=1):
        numbered_events.append(f"{idx}. {timestamp[0]:.1f}s-{timestamp[1]:.1f}s: {sentence}")
    events_text = "\n".join(numbered_events)
    return f"""Create one global description for the whole video.

Requirements:
- Use 40 to 70 words.
- Focus on visible actors, actions, interactions, objects, and temporal progression.
- Do not mention the class name: {record['class_name']}.
- Do not use words from this forbidden list: {', '.join(FORBIDDEN_LABELS)}.
- Do not add any event that is not in the timestamp descriptions.
- Use plain ASCII punctuation only.
- Return exactly this JSON shape: {{"description": "..."}}

Video ID: {record['video_id']}
Duration: {record['duration']:.2f} seconds
Timestamp descriptions:
{events_text}"""


def extract_json_description(text):
    if not text:
        return "", "empty_response"
    stripped = normalize_text_artifacts(text).strip()
    try:
        payload = json.loads(stripped)
        return normalize_text_artifacts(str(payload.get("description", "")).strip()), "ok"
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", stripped, flags=re.DOTALL)
    if match:
        try:
            payload = json.loads(match.group(0))
            return normalize_text_artifacts(str(payload.get("description", "")).strip()), "recovered_json"
        except json.JSONDecodeError:
            pass
    return normalize_text_artifacts(stripped), "raw_text"


def count_words(text):
    return len(re.findall(r"\b[\w'-]+\b", text))


def find_forbidden_labels(text):
    text_norm = text.lower()
    found = []
    for label in FORBIDDEN_LABELS:
        pattern = r"\b" + re.escape(label).replace(r"\ ", r"\s+") + r"s?\b"
        if re.search(pattern, text_norm):
            found.append(label)
    return found


def validate_description(description, parse_status="ok"):
    word_count = count_words(description)
    forbidden = find_forbidden_labels(description)
    non_ascii_chars = sorted({ch for ch in description if ord(ch) > 127})
    issues = []
    if not description.strip():
        issues.append("empty")
    if word_count < 40:
        issues.append("too_short")
    if word_count > 70:
        issues.append("too_long")
    if forbidden:
        issues.append("contains_forbidden_label")
    if non_ascii_chars:
        issues.append("contains_non_ascii")
    if parse_status not in {"ok", "recovered_json"}:
        issues.append(f"parse_{parse_status}")
    return {
        "word_count": word_count,
        "contains_forbidden_label": bool(forbidden),
        "forbidden_labels": forbidden,
        "contains_non_ascii": bool(non_ascii_chars),
        "non_ascii_chars": non_ascii_chars,
        "status": "ok" if not issues else "warning",
        "issues": issues,
    }


print(make_user_prompt(records_to_process[0])[:1200])


Create one global description for the whole video.

Requirements:
- Use 40 to 70 words.
- Focus on visible actors, actions, interactions, objects, and temporal progression.
- Do not mention the class name: Abuse.
- Do not use words from this forbidden list: abuse, arrest, arson, assault, burglary, explosion, fighting, road accident, roadaccident, robbery, shooting, shoplifting, stealing, vandalism, anomaly, anomalous, crime.
- Do not add any event that is not in the timestamp descriptions.
- Use plain ASCII punctuation only.
- Return exactly this JSON shape: {"description": "..."}

Video ID: Abuse001_x264
Duration: 91.00 seconds
Timestamp descriptions:
1. 0.0s-5.3s: A woman with short hair, slightly fat, wearing a white top and black pants stood in front of the table, picked up a book from the table, and opened it to read
2. 7.0s-8.5s: A man wearing a white shirt and black pants entered the house and walked towards the short-haired and fat woman in front who was reading a book.
3. 7.2s

## 5. OpenAI API Helpers


In [5]:
def api_key_is_filled(value, placeholder):
    return bool(value and value.strip() and value.strip() != placeholder)


def extract_openai_output_text(response_json):
    if response_json.get("output_text"):
        return normalize_text_artifacts(response_json["output_text"])
    chunks = []

    def visit(value):
        if isinstance(value, dict):
            value_type = value.get("type")
            if value_type in {"output_text", "text"} and isinstance(value.get("text"), str):
                chunks.append(value["text"])
            if isinstance(value.get("content"), str):
                chunks.append(value["content"])
            for child in value.values():
                visit(child)
        elif isinstance(value, list):
            for child in value:
                visit(child)

    visit(response_json.get("output", []))
    return normalize_text_artifacts("\n".join(chunk for chunk in chunks if chunk).strip())


def get_response_debug(response_json):
    if not isinstance(response_json, dict):
        return {}
    return {
        "id": response_json.get("id"),
        "model": response_json.get("model"),
        "status": response_json.get("status"),
        "finish_reason": response_json.get("finish_reason"),
        "incomplete_details": response_json.get("incomplete_details"),
        "error": response_json.get("error"),
    }


def call_openai_description(record):
    if DRY_RUN:
        return "", {"skipped": True, "reason": "DRY_RUN is True"}
    if not api_key_is_filled(OPENAI_API_KEY, "PASTE_OPENAI_API_KEY_HERE"):
        return "", {"skipped": True, "reason": "OPENAI_API_KEY is not filled"}

    url = "https://api.openai.com/v1/responses"
    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": OPENAI_MODEL,
        "input": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": make_user_prompt(record)},
        ],
        "max_output_tokens": OPENAI_MAX_OUTPUT_TOKENS,
    }
    response = requests.post(url, headers=headers, json=payload, timeout=REQUEST_TIMEOUT_SECONDS)
    try:
        response_json = response.json()
    except ValueError:
        response.raise_for_status()
        raise RuntimeError(f"OpenAI returned non-JSON response: {response.text[:500]}")
    if response.status_code >= 400:
        raise RuntimeError(f"OpenAI API error {response.status_code}: {json.dumps(response_json)[:1000]}")
    return extract_openai_output_text(response_json), response_json


def generate_with_retries(record, call_fn):
    attempts = []
    last_description = ""
    last_parse_status = "empty_response"
    last_validation = validate_description("", last_parse_status)
    last_raw_text = ""
    last_response_debug = {}

    for attempt_idx in range(1, MAX_RETRIES + 1):
        try:
            raw_text, response_json = call_fn(record)
            description, parse_status = extract_json_description(raw_text)
            validation = validate_description(description, parse_status)
            response_debug = get_response_debug(response_json)
            attempt = {
                "attempt": attempt_idx,
                "parse_status": parse_status,
                "validation": validation,
                "raw_text": raw_text,
                "response_debug": response_debug,
            }
            attempts.append(attempt)
            last_description = description
            last_parse_status = parse_status
            last_validation = validation
            last_raw_text = raw_text
            last_response_debug = response_debug

            if response_json.get("skipped"):
                break
            if validation["status"] == "ok":
                break
        except Exception as exc:
            attempts.append({"attempt": attempt_idx, "error": repr(exc)})
            last_parse_status = "error"
            last_validation = validate_description("", last_parse_status)
            if "403" in repr(exc) or "401" in repr(exc):
                break

        if attempt_idx < MAX_RETRIES:
            time.sleep(RETRY_BACKOFF_SECONDS * attempt_idx)

    return {
        "description": last_description,
        "parse_status": last_parse_status,
        "validation": last_validation,
        "raw_text": last_raw_text,
        "response_debug": last_response_debug,
        "attempts": attempts,
    }


print("OpenAI key filled:", api_key_is_filled(OPENAI_API_KEY, "PASTE_OPENAI_API_KEY_HERE"))


OpenAI key filled: True


## 6. Generate Or Rerun GPT Descriptions


In [6]:
existing_results = []
existing_by_key = {}
records_to_generate = records_to_process

if RERUN_WARNINGS_ONLY:
    if not OUTPUT_JSON.exists():
        raise FileNotFoundError(f"Cannot rerun warnings because {OUTPUT_JSON} does not exist.")
    with OUTPUT_JSON.open("r", encoding="utf-8") as f:
        existing_results = json.load(f)
    existing_by_key = {(record["split"], record["video_id"]): record for record in existing_results}
    warning_keys = {
        key
        for key, record in existing_by_key.items()
        if record.get("gpt_validation", {}).get("status") != "ok"
    }
    records_to_generate = [
        record for record in records_to_process
        if (record["split"], record["video_id"]) in warning_keys
    ]
    print("Existing records:", len(existing_results))
    print("Warning records to rerun:", len(records_to_generate))

new_results = []

for idx, record in enumerate(records_to_generate, start=1):
    print(f"[{idx}/{len(records_to_generate)}] {record['split']} | {record['class_name']} | {record['video_id']}")
    result = dict(record)

    openai_result = generate_with_retries(record, call_openai_description)
    provider_errors = {}
    errors = [attempt.get("error") for attempt in openai_result["attempts"] if attempt.get("error")]
    if errors:
        provider_errors["openai"] = errors[-1]

    result["gpt_description"] = openai_result["description"]
    result["gpt_validation"] = openai_result["validation"]
    result["gpt_parse_status"] = openai_result["parse_status"]
    result["gpt_raw_text"] = openai_result["raw_text"]
    result["gpt_response_debug"] = openai_result["response_debug"]
    result["gpt_attempts"] = openai_result["attempts"]
    result["provider_errors"] = provider_errors
    new_results.append(result)

    time.sleep(SLEEP_BETWEEN_REQUESTS_SECONDS)

if RERUN_WARNINGS_ONLY:
    for result in new_results:
        existing_by_key[(result["split"], result["video_id"])] = result
    split_order = {split_name: idx for idx, split_name in enumerate(SPLITS_TO_PROCESS)}
    results = sorted(
        existing_by_key.values(),
        key=lambda item: (split_order[item["split"]], item["class_name"], item["video_id"]),
    )
    print("Updated warning records:", len(new_results))
    print("Merged records:", len(results))
else:
    results = new_results
    print("Generated records:", len(results))


Existing records: 1854
Warning records to rerun: 22
[1/22] train | Abuse | Abuse005_x264
[2/22] train | Arrest | Arrest004_x264
[3/22] train | Assault | Assault016_x264
[4/22] train | Normal | Normal_Videos092_x264
[5/22] train | Normal | Normal_Videos180_x264
[6/22] train | Normal | Normal_Videos186_x264
[7/22] train | Normal | Normal_Videos197_x264
[8/22] train | Normal | Normal_Videos207_x264
[9/22] train | Normal | Normal_Videos291_x264
[10/22] train | Normal | Normal_Videos363_x264
[11/22] train | Normal | Normal_Videos422_x264
[12/22] train | Normal | Normal_Videos456_x264
[13/22] train | Normal | Normal_Videos_641_x264
[14/22] train | Normal | Normal_Videos_896_x264
[15/22] train | Robbery | Robbery006_x264
[16/22] train | Robbery | Robbery018_x264
[17/22] train | Shoplifting | Shoplifting007_x264
[18/22] val | Vandalism | Vandalism050_x264
[19/22] test | Robbery | Robbery086_x264
[20/22] test | Robbery | Robbery100_x264
[21/22] test | Shoplifting | Shoplifting034_x264
[22/22] t

## 7. Review Validation Summary


In [7]:
status_counts = Counter(result["gpt_validation"]["status"] for result in results)
issue_counts = Counter()
for result in results:
    issue_counts.update(result["gpt_validation"]["issues"])

print("GPT")
print("  statuses:", dict(status_counts))
print("  issues:", dict(issue_counts))
print("  non-empty:", sum(bool(result["gpt_description"].strip()) for result in results), "/", len(results))

print("\nPreview:")
for result in results[:5]:
    print("-", result["split"], "|", result["video_id"], "|", result["class_name"])
    print("  raw:", result["raw_joined_description"][:180], "...")
    print("  gpt:", result["gpt_description"][:220])


GPT
  statuses: {'ok': 1843, 'warning': 11}
  issues: {'too_long': 6, 'empty': 4, 'too_short': 5, 'parse_empty_response': 4, 'parse_raw_text': 1}
  non-empty: 1850 / 1854

Preview:
- train | Abuse001_x264 | Abuse
  raw: A woman with short hair, slightly fat, wearing a white top and black pants stood in front of the table, picked up a book from the table, and opened it to read A man wearing a white ...
  gpt: A short-haired woman in a white top and black pants reads a book at a table. Two men enter and approach her; one pulls a red cloth from her side and runs away, while the other strikes her head and leaves. She falls, drop
- train | Abuse002_x264 | Abuse
  raw: At an intersection with smooth traffic, the green light is on, and the vehicles on the opposite side drive out at the same time. There are many white cars and many passers-by on bi ...
  gpt: At a busy intersection, vehicles, bicycles, and electric vehicles move through a green light. A child falls from the open trunk of a sil

## 8. Save Results


In [8]:
CODE_DIR.mkdir(parents=True, exist_ok=True)

with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

csv_fields = [
    "split",
    "video_id",
    "class_name",
    "duration",
    "raw_joined_description",
    "gpt_description",
    "gpt_word_count",
    "gpt_status",
    "gpt_issues",
    "gpt_parse_status",
    "gpt_attempt_count",
    "gpt_raw_text",
    "provider_errors",
]

with OUTPUT_CSV.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields)
    writer.writeheader()
    for result in results:
        writer.writerow(
            {
                "split": result["split"],
                "video_id": result["video_id"],
                "class_name": result["class_name"],
                "duration": result["duration"],
                "raw_joined_description": result["raw_joined_description"],
                "gpt_description": result["gpt_description"],
                "gpt_word_count": result["gpt_validation"]["word_count"],
                "gpt_status": result["gpt_validation"]["status"],
                "gpt_issues": ";".join(result["gpt_validation"]["issues"]),
                "gpt_parse_status": result["gpt_parse_status"],
                "gpt_attempt_count": len(result["gpt_attempts"]),
                "gpt_raw_text": result["gpt_raw_text"],
                "provider_errors": json.dumps(result["provider_errors"], ensure_ascii=False),
            }
        )

print("Saved JSON:", OUTPUT_JSON)
print("Saved CSV:", OUTPUT_CSV)


Saved JSON: D:\Finetune VadCLIP\code\ucf_gpt_video_descriptions.json
Saved CSV: D:\Finetune VadCLIP\code\ucf_gpt_video_descriptions.csv
